# JobShield – 02 Preprocessing (Week 2): EDA part 2, cleaning and data split

**Group 5 · AML-3403 AI and ML Capstone · Week 2 lead: Arti Makwana**

1. Finish EDA: categorical fields and location
2. Missing-value strategy and text preprocessing
3. Stratified 70/15/15 train / validation / test split
4. Evaluation functions and the "always real" baseline
5. Summary for Weekly Report 2

Needs `data/raw/fake_job_postings.csv` (see README). In Colab run `!pip install nltk` first if needed.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.data.loader import load_emscad, TARGET
from src.data.split import make_splits, split_summary, save_splits
from src.features.text_cleaning import preprocess
from src.evaluation.metrics import evaluate, print_report, log_result, plot_confusion
from src.utils.config import load_config, project_path, set_seed
CFG = load_config(); set_seed(CFG["seed"])

DATA_PATH = project_path(CFG["paths"]["raw_data"])
FIG_DIR = project_path(CFG["paths"]["figures_dir"]); FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
FAKE = "#C55A11"
summary = {}

raw = load_emscad(DATA_PATH)
print(f"Loaded {len(raw):,} postings")

## 1. EDA part 2 – fraud rate by categorical field
Only groups with at least 50 postings are shown, so rates are not based on a handful of examples.

In [ ]:
def fraud_rate_table(df, col, min_count=CFG["eda"]["min_group_size"]):
    g = df.groupby(df[col].fillna("Missing"))[TARGET].agg(postings="count", fake="sum")
    g["fraud_rate_%"] = (g["fake"] / g["postings"] * 100).round(2)
    return g[g["postings"] >= min_count].sort_values("fraud_rate_%", ascending=False)

overall = raw[TARGET].mean() * 100
cols = ["employment_type", "required_experience", "required_education"]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, col in zip(axes, cols):
    t = fraud_rate_table(raw, col).head(8)
    ax.barh(t.index[::-1], t["fraud_rate_%"][::-1], color=FAKE)
    ax.axvline(overall, ls="--", color="grey", lw=1)
    ax.set_title(col.replace("_", " ").title()); ax.set_xlabel("Fraud rate (%)")
    summary[f"highest_fraud_{col}"] = f"{t.index[0]} ({t['fraud_rate_%'].iloc[0]}%)"
plt.tight_layout(); plt.savefig(FIG_DIR / "06_fraud_rate_categorical.png", dpi=200); plt.show()

In [ ]:
for col in ["industry", "function"]:
    t = fraud_rate_table(raw, col).head(10)
    summary[f"highest_fraud_{col}"] = f"{t.index[0]} ({t['fraud_rate_%'].iloc[0]}%)"
    print(f"\n=== Top 10 {col} by fraud rate ===")
    print(t.to_string())

### Location (country)

In [ ]:
raw["country"] = raw["location"].fillna("Missing").str.split(",").str[0].str.strip().replace("", "Missing")
ct = fraud_rate_table(raw, "country", min_count=100)
summary["countries_with_100plus"] = len(ct)
summary["highest_fraud_country"] = f"{ct.index[0]} ({ct['fraud_rate_%'].iloc[0]}%)"
print(ct.head(12).to_string())

top = raw["country"].value_counts().head(10).index
t = fraud_rate_table(raw[raw["country"].isin(top)], "country", min_count=1).sort_values("postings")
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.barh(t.index, t["fraud_rate_%"], color=FAKE)
for y, (r, n) in enumerate(zip(t["fraud_rate_%"], t["postings"])):
    ax.text(r, y, f"  {r:.1f}%  (n={n:,})", va="center", fontsize=8)
ax.axvline(overall, ls="--", color="grey", lw=1)
ax.set_xlabel("Fraud rate (%)"); ax.set_title("Fraud rate in the 10 most common countries")
plt.tight_layout(); plt.savefig(FIG_DIR / "07_fraud_rate_country.png", dpi=200); plt.show()

## 2. Missing-value strategy and preprocessing

**Team decision (Week 2):** missing information is itself a warning sign (EDA in Week 1 showed fake postings
leave fields such as company profile empty more often), so we **keep it as a signal** instead of dropping rows:

| Field type | Strategy |
|---|---|
| Text (company profile, description, requirements, benefits) | Fill with empty text **and** add a `missing_*` 0/1 flag |
| Categorical (employment type, industry, …) | Fill with the category `"Missing"` |
| Rows | No rows dropped for missing values; only exact duplicate postings are removed |

**Two cleaned text versions:** `text_clean` (lower-case, HTML/links/e-mails/phones replaced by tokens, lemmatised,
stop words removed) for TF-IDF models, and `text_bert` (light cleaning, case and punctuation kept) for DistilBERT.

In [ ]:
clean = preprocess(raw, lemmatize=CFG["preprocessing"]["lemmatize"], drop_duplicates=CFG["preprocessing"]["drop_duplicates"])
summary.update(clean.attrs["preprocess_stats"])
interim = project_path(CFG["paths"]["interim_data"]); interim.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(interim, index=False)
print(f"Saved cleaned data to {interim.relative_to(ROOT)}")

has_desc = clean[clean["missing_description"] == 0]
i = has_desc["description"].str.len().sub(500).abs().idxmin()     # a typical posting with a description
print("BEFORE  raw description (first 300 chars):\n", clean.loc[i, "description"][:300])
print("\nAFTER   text_bert (first 300 chars):\n", clean.loc[i, "text_bert"][:300])
print("\nAFTER   text_clean (first 300 chars):\n", clean.loc[i, "text_clean"][:300])

In [ ]:
flag_cols = [c for c in clean.columns if c.startswith("missing_")]
mf = clean.groupby(TARGET)[flag_cols].mean().T * 100
mf.columns = ["real_%", "fake_%"]
print("Share of postings with the field missing (%):")
print(mf.round(1).sort_values("fake_%", ascending=False).to_string())

## 3. Stratified 70 / 15 / 15 split (ratios and seed from configs/config.yaml)

In [ ]:
train_df, val_df, test_df = make_splits(clean)
st = split_summary({"train": train_df, "val": val_df, "test": test_df})
print(st.to_string(index=False))
summary["split_rows"] = dict(zip(st.split, st.rows))
summary["split_fake"] = dict(zip(st.split, st.fake))

# sanity checks: no job appears in two splits, fake ratio equal everywhere
ids = [set(p.job_id) for p in (train_df, val_df, test_df)]
assert not (ids[0] & ids[1] or ids[0] & ids[2] or ids[1] & ids[2]), "job_id overlap between splits!"
assert st["fake_%"].max() - st["fake_%"].min() < 0.5, "fake ratio differs between splits"
print("Checks passed: no overlap between splits, same fake ratio in each.")
save_splits(train_df, val_df, test_df)

## 4. Evaluation functions – why accuracy is misleading
Two dummy baselines on the validation set, scored with `src/evaluate.py`.

In [ ]:
rng = np.random.default_rng(42)
y = val_df[TARGET].values

always_real = evaluate(y, np.zeros_like(y), y_score=np.zeros(len(y)), model_name="Always predict real", split="val")
rand_score = rng.random(len(y))
random_guess = evaluate(y, (rand_score < y.mean()).astype(int), y_score=rand_score, model_name="Random guess", split="val")

for r in (always_real, random_guess):
    print_report(r); log_result(r)
summary["always_real_accuracy"] = round(always_real["accuracy"] * 100, 2)
summary["always_real_recall"] = always_real["recall"]

plot_confusion(always_real, save_as="08_always_real_confusion.png"); plt.show()
pd.DataFrame([always_real, random_guess])[["model", "precision", "recall", "f1", "pr_auc", "accuracy"]].round(3)

## 5. Summary for Weekly Report 2

In [ ]:
for k, v in summary.items():
    print(f"{k:32s}: {v}")